# Qwen3.5-0.8B Fine-tuning

**Colab secrets needed:**
- `HF_TOKEN` (required)
- `WANDB_API_KEY` (optional, for logging)

**Runtime:** T4 free tier (~10-12GB VRAM with QLoRA)

In [ ]:
!pip install transformers accelerate bitsandbytes peft trl huggingface_hub wandb -q

In [ ]:
import os
from google.colab import userdata

HF_TOKEN = os.environ.get('HF_TOKEN') or userdata.get('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN not found in Colab Secrets')
print(f'HF_TOKEN: {HF_TOKEN[:8]}...')

WANDB_KEY = os.environ.get('WANDB_API_KEY') or userdata.get('WANDB_API_KEY')
if WANDB_KEY:
    import wandb
    wandb.login(key=WANDB_KEY)
    wandb.init(project='qwen-finetune', name='qwen-0.8b-starfire')
    print('wandb initialized')
else:
    print('WANDB_API_KEY not found - logging disabled')

from huggingface_hub import login
login(token=HF_TOKEN)

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## CONFIG

In [ ]:
from dataclasses import dataclass

@dataclass
class C:
    model_name: str = 'Qwen/Qwen3.5-0.8B-Base'
    load_in_4bit: bool = True
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    batch_size: int = 2
    grad_accum: int = 8
    epochs: int = 3
    lr: float = 2e-4
    max_seq: int = 512
    dataset: str = 'ZacharyMaronek/starfire-personal-v1'
    output_dir: str = './qwen-finetuned'

cfg = C()
print(f'Model: {cfg.model_name}')
print(f'Effective batch: {cfg.batch_size * cfg.grad_accum}')

## LOAD DATASET

In [ ]:
from datasets import load_dataset

ds = load_dataset(cfg.dataset, split='train')
print(f'Loaded {len(ds)} examples')
print(f'Columns: {ds.column_names}')
print(f'First example: {ds[0]}')

## FORMAT + TOKENIZE

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

def format_fn(example):
    user_text = example.get('input', '')
    assistant_text = example.get('output', '')
    text = '<|user|> ' + str(user_text) + ' <|end|><|assistant|> ' + str(assistant_text) + ' <|end|>'
    enc = tokenizer(text, max_length=cfg.max_seq, truncation=True, padding='max_length')
    enc['labels'] = enc['input_ids'].copy()
    return enc

formatted = ds.map(format_fn, remove_columns=ds.column_names)
split = formatted.train_test_split(test_size=0.05, seed=42)
train_ds, val_ds = split['train'], split['test']
print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')

## LOAD MODEL - QLoRA

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print('Loading model with QLoRA...')
model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    quantization_config=quant_config,
    device_map='auto',
    trust_remote_code=True
)
print(f'Memory: {model.get_memory_footprint()/1e9:.1f} GB')

## ADD LoRA ADAPTERS

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_cfg = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=cfg.lora_dropout,
    bias='none',
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()
model.gradient_checkpointing_enable()
model.config.use_cache = False

## TRAIN

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.epochs,
    per_device_train_batch_size=cfg.batch_size,
    gradient_accumulation_steps=cfg.grad_accum,
    learning_rate=cfg.lr,
    warmup_steps=100,
    logging_steps=10,
    save_steps=200,
    eval_steps=200,
    eval_strategy='steps',
    save_strategy='steps',
    bf16=True,
    optim='paged_adamw_8bit',
    report_to='wandb' if WANDB_KEY else 'none',
    seed=42
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    max_seq_length=cfg.max_seq
)

print('Starting training...')
trainer.train()
print('Training complete!')

## EVALUATE

In [ ]:
import math

results = trainer.evaluate()
print(f"Eval loss: {results['eval_loss']:.4f}")
print(f"Perplexity: {math.exp(results['eval_loss']):.2f}")

if WANDB_KEY:
    wandb.log({'eval_loss': results['eval_loss'], 'perplexity': math.exp(results['eval_loss'])})
    wandb.finish()

## SAVE + MERGE

In [ ]:
trainer.save_model(cfg.output_dir)
print(f'Adapter saved: {cfg.output_dir}')

from peft import PeftModel
from transformers import AutoModelForCausalLM

merged_dir = f'{cfg.output_dir}-merged'

print('Merging LoRA with base model...')
base = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    torch_dtype=torch.bfloat16,
    device_map='cpu',
    trust_remote_code=True
)
merged = PeftModel.from_pretrained(base, cfg.output_dir).merge_and_unload()
merged.save_pretrained(merged_dir)
tokenizer.save_pretrained(merged_dir)
print(f'Merged model: {merged_dir}')

## TEST GENERATION

In [ ]:
from transformers import pipeline

merged_dir = f'{cfg.output_dir}-merged'
pipe = pipeline('text-generation', model=merged_dir, tokenizer=tokenizer, device_map='auto')

prompts = [
    '<|user|> Hello, how are you? <|end|><|assistant|>',
    '<|user|> Tell me about starfire. <|end|><|assistant|>',
]

for prompt in prompts:
    print(f'PROMPT: {prompt}')
    result = pipe(prompt, max_new_tokens=64, temperature=0.7, do_sample=True)
    print(f"OUTPUT: {result[0]['generated_text']}")
    print()